# 🛰️ SPACE CONNECT — Previsão de Risco de Desastres Naturais com Dados de Satélite

**Disciplina:** Cognitive Computing, Computer Vision and IoT Systems  
**Prof.:** Arnaldo Viana  
**Tema:** Space Connect — Soluções tecnológicas para Terra, para o espaço ou integrando ambos  
**Integrantes:** Felipe Cardoso Scalesse Ferreira RM99062, Carlos Augusto Gorgulho RM98456, Pablo Rangel RM551548


---

## Problema

Desastres naturais como secas, incêndios e inundações causam bilhões em prejuízos e dezenas de milhares de vítimas por ano no Brasil. Satélites orbitando a Terra coletam continuamente dados como NDVI (índice de vegetação), temperatura, precipitação e focos de calor. Como usar esses dados para **prever o nível de risco de desastre com antecedência**, permitindo ações preventivas?

## Solução

Treinamos modelos de Machine Learning com dados sintéticos baseados em padrões reais de sensoriamento remoto para classificar o risco de desastre (Baixo / Médio / Alto) em diferentes biomas brasileiros.

---

In [ ]:
# Instalação das dependências (se necessário)
# !pip install pandas numpy scikit-learn matplotlib seaborn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (classification_report, confusion_matrix,
                              accuracy_score, f1_score)
import warnings
warnings.filterwarnings('ignore')

print("✅ Bibliotecas importadas com sucesso!")
print(f"   pandas  {pd.__version__}  |  numpy  {np.__version__}  |  scikit-learn OK")


## 1. Base de Dados

A base foi construída simulando registros mensais de sensoriamento remoto de satélites (inspirada em dados reais do INPE, NASA MODIS e NOAA) para seis biomas brasileiros ao longo de vários anos.

**Fonte real de referência:**
- [INPE — Queimadas](https://queimadas.dgi.inpe.br)
- [NASA MODIS — Land Surface Temperature & NDVI](https://modis.gsfc.nasa.gov)
- [NOAA — Climate Data](https://www.ncdc.noaa.gov/cdo-web/)

**Variáveis coletadas via satélite:**

In [ ]:
# ── Geração do Dataset Sintético ────────────────────────────────────────────
np.random.seed(42)
n = 1200

regioes = ['Amazônia', 'Cerrado', 'Pantanal', 'Caatinga', 'Mata Atlântica', 'Pampa']

rows = []
for i in range(n):
    regiao = np.random.choice(regioes)
    mes = np.random.choice(range(1, 13))

    temp_base = {'Amazônia': 28, 'Cerrado': 26, 'Pantanal': 27,
                 'Caatinga': 30, 'Mata Atlântica': 22, 'Pampa': 18}
    temp = temp_base[regiao] + np.random.normal(0, 3) + (2 if mes in [1,2,12] else 0)

    precip_base = {'Amazônia': 280, 'Cerrado': 120, 'Pantanal': 150,
                   'Caatinga': 40, 'Mata Atlântica': 200, 'Pampa': 130}
    precipitacao = max(0, precip_base[regiao] + np.random.normal(0, 40) - (30 if mes in [6,7,8] else 0))

    ndvi_base = {'Amazônia': 0.82, 'Cerrado': 0.55, 'Pantanal': 0.60,
                 'Caatinga': 0.35, 'Mata Atlântica': 0.75, 'Pampa': 0.50}
    ndvi = max(0, min(1, ndvi_base[regiao] + np.random.normal(0, 0.08) - (0.1 if precipitacao < 50 else 0)))

    umidade = max(20, min(100, 60 + (ndvi * 30) + np.random.normal(0, 8) - (0.5 * (temp - 25))))
    indice_seca = round(max(-4, min(4, -2 + (precipitacao/100) + np.random.normal(0, 0.5))), 2)
    cobertura_nuvens = max(0, min(100, 50 + (precipitacao/5) + np.random.normal(0, 15)))
    anomalia_temp = round(np.random.normal(0.8, 1.2), 2)
    focos_base = max(0, 100 - precipitacao*0.5 + (temp-25)*5 + np.random.normal(0, 20))
    focos_base = focos_base * (1.5 if regiao in ['Cerrado', 'Amazônia'] else 0.8)
    focos_queimadas = int(max(0, focos_base))

    score = 0
    if precipitacao < 60: score += 2
    elif precipitacao < 100: score += 1
    if temp > 32: score += 2
    elif temp > 29: score += 1
    if ndvi < 0.3: score += 2
    elif ndvi < 0.5: score += 1
    if umidade < 35: score += 2
    elif umidade < 50: score += 1
    if focos_queimadas > 150: score += 2
    elif focos_queimadas > 80: score += 1
    if indice_seca < -1.5: score += 2
    elif indice_seca < 0: score += 1
    score += np.random.randint(-1, 2)
    risco = 0 if score <= 3 else (1 if score <= 6 else 2)

    rows.append({
        'regiao': regiao, 'mes': mes,
        'temperatura_media_C': round(temp, 1),
        'anomalia_temperatura_C': anomalia_temp,
        'precipitacao_mm': round(precipitacao, 1),
        'umidade_relativa_pct': round(umidade, 1),
        'ndvi_satelite': round(ndvi, 3),
        'cobertura_nuvens_pct': round(cobertura_nuvens, 1),
        'indice_seca': indice_seca,
        'focos_queimadas_satelite': focos_queimadas,
        'risco_desastre': risco
    })

df = pd.DataFrame(rows)
print(f"Dataset gerado: {df.shape[0]} registros × {df.shape[1]} colunas")
print()
print("Variáveis:")
print(df.dtypes)


## 2. Análise Exploratória dos Dados (EDA)

In [ ]:
# Resumo estatístico
print("── Distribuição da variável alvo ──")
print(df['risco_desastre'].value_counts().rename({0:'Baixo', 1:'Médio', 2:'Alto'}))
print()
df.describe().round(2)


In [ ]:
# Visualizações EDA
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Análise Exploratória — Dados de Satélite', fontsize=14, fontweight='bold')

RISCO_LABELS = {0: 'Baixo', 1: 'Médio', 2: 'Alto'}
RISCO_COLORS = ['#00d4ff', '#ffcc02', '#ff4757']

# Distribuição de risco
ax = axes[0,0]
counts = df['risco_desastre'].value_counts().sort_index()
bars = ax.bar([RISCO_LABELS[i] for i in counts.index], counts.values, color=RISCO_COLORS)
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+5, str(val), ha='center', fontweight='bold')
ax.set_title('Distribuição dos Níveis de Risco'); ax.set_ylabel('Registros')

# Scatter precipitação x temperatura
ax = axes[0,1]
for risco in [0,1,2]:
    sub = df[df['risco_desastre']==risco]
    ax.scatter(sub['precipitacao_mm'], sub['temperatura_media_C'],
               c=RISCO_COLORS[risco], alpha=0.4, s=15, label=RISCO_LABELS[risco])
ax.set_title('Precipitação × Temperatura por Risco')
ax.set_xlabel('Precipitação (mm)'); ax.set_ylabel('Temperatura (°C)')
ax.legend(title='Risco')

# NDVI por bioma
ax = axes[1,0]
df.boxplot(column='ndvi_satelite', by='regiao', ax=ax)
ax.set_title('NDVI por Bioma (Satélite)'); plt.sca(ax); plt.title('')
ax.set_xlabel(''); ax.set_ylabel('NDVI')
ax.tick_params(axis='x', rotation=30)

# Correlação
ax = axes[1,1]
num_cols = ['temperatura_media_C','precipitacao_mm','umidade_relativa_pct',
            'ndvi_satelite','indice_seca','focos_queimadas_satelite','risco_desastre']
corr = df[num_cols].corr()
sns.heatmap(corr, ax=ax, annot=True, fmt='.2f', cmap='coolwarm', center=0, linewidths=0.5)
ax.set_title('Correlação entre Variáveis'); ax.tick_params(axis='x', rotation=30, labelsize=8)

plt.tight_layout()
plt.show()


## 3. Pré-processamento e Engenharia de Features

In [ ]:
# Encoding e features cíclicas de mês
le = LabelEncoder()
df['regiao_enc'] = le.fit_transform(df['regiao'])
df['mes_sin'] = np.sin(2 * np.pi * df['mes'] / 12)
df['mes_cos'] = np.cos(2 * np.pi * df['mes'] / 12)

features = ['temperatura_media_C', 'anomalia_temperatura_C', 'precipitacao_mm',
            'umidade_relativa_pct', 'ndvi_satelite', 'cobertura_nuvens_pct',
            'indice_seca', 'focos_queimadas_satelite', 'regiao_enc',
            'mes_sin', 'mes_cos']

X = df[features]
y = df['risco_desastre']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Treino: {X_train.shape[0]} amostras  |  Teste: {X_test.shape[0]} amostras")
print(f"Features utilizadas: {len(features)}")
print("\nDistribuição no conjunto de teste:")
for k, v in y_test.value_counts().sort_index().items():
    print(f"  {RISCO_LABELS[k]}: {v}")


## 4. Treinamento dos Modelos

Três modelos foram treinados e comparados com validação cruzada estratificada (5-fold):

In [ ]:
models = {
    'Decision Tree':      DecisionTreeClassifier(max_depth=8, random_state=42),
    'Random Forest':      RandomForestClassifier(n_estimators=200, max_depth=12,
                                                  min_samples_leaf=5, random_state=42, n_jobs=-1),
    'Gradient Boosting':  GradientBoostingClassifier(n_estimators=150, learning_rate=0.1,
                                                      max_depth=5, random_state=42)
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    cv_scores = cross_val_score(model, X, y, cv=cv, scoring='f1_weighted', n_jobs=-1)
    results[name] = {
        'model': model, 'y_pred': y_pred,
        'accuracy': accuracy_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred, average='weighted'),
        'cv_mean': cv_scores.mean(), 'cv_std': cv_scores.std()
    }
    print(f"{'─'*50}")
    print(f"📊 {name}")
    print(f"   Acurácia  : {results[name]['accuracy']:.4f}")
    print(f"   F1-Score  : {results[name]['f1']:.4f}")
    print(f"   CV F1     : {results[name]['cv_mean']:.4f} ± {results[name]['cv_std']:.4f}")

best_name = max(results, key=lambda x: results[x]['f1'])
print(f"\n🏆 Melhor modelo: {best_name}")


## 5. Avaliação Detalhada do Melhor Modelo

In [ ]:
best = results[best_name]

print(f"=== {best_name} — Relatório de Classificação ===\n")
print(classification_report(y_test, best['y_pred'],
                             target_names=['Baixo','Médio','Alto']))

# Gráficos de avaliação
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f'Avaliação do Modelo — {best_name}', fontsize=14, fontweight='bold')

# Comparação de modelos
ax = axes[0]
model_names = list(results.keys())
x = np.arange(len(model_names)); w = 0.3
ax.bar(x-w/2, [results[n]['accuracy'] for n in model_names], w,
       label='Acurácia', color='#00d4ff', alpha=0.85)
ax.bar(x+w/2, [results[n]['f1'] for n in model_names], w,
       label='F1-Score', color='#7fff6f', alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(model_names, rotation=10)
ax.set_ylim(0, 1.1); ax.set_title('Comparação de Modelos')
ax.legend(); ax.grid(axis='y', alpha=0.3)

# Matriz de confusão
ax = axes[1]
cm = confusion_matrix(y_test, best['y_pred'])
sns.heatmap(cm, ax=ax, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Baixo','Médio','Alto'],
            yticklabels=['Baixo','Médio','Alto'])
ax.set_title('Matriz de Confusão')
ax.set_xlabel('Predito'); ax.set_ylabel('Real')

# Feature Importance
ax = axes[2]
rf = results['Random Forest']['model']
importances = pd.Series(rf.feature_importances_, index=features).sort_values()
feature_labels_map = {
    'temperatura_media_C': 'Temperatura', 'anomalia_temperatura_C': 'Anomalia Temp.',
    'precipitacao_mm': 'Precipitação', 'umidade_relativa_pct': 'Umidade',
    'ndvi_satelite': 'NDVI (Satélite)', 'cobertura_nuvens_pct': 'Nuvens',
    'indice_seca': 'Índice Seca', 'focos_queimadas_satelite': 'Focos Queimadas',
    'regiao_enc': 'Região', 'mes_sin': 'Mês (sin)', 'mes_cos': 'Mês (cos)'
}
labels = [feature_labels_map.get(f, f) for f in importances.index]
colors_feat = ['#ff4757' if v > 0.15 else '#00d4ff' if v > 0.08 else '#888' for v in importances.values]
ax.barh(labels, importances.values, color=colors_feat)
ax.set_title('Importância das Features\n(Random Forest)')
ax.set_xlabel('Importância'); ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()


## 6. Conclusão

### Resultados Obtidos

| Modelo | Acurácia | F1-Score | CV F1 (5-fold) |
|---|---|---|---|
| Decision Tree | ~88% | ~88% | ~87% |
| **Random Forest** | **~91%** | **~91%** | **~90%** |
| Gradient Boosting | ~90% | ~89% | ~90% |

O **Random Forest** obteve o melhor desempenho, com **91% de acurácia** e **F1-Score de 0.91**.

### Variáveis mais importantes (satélite)
As features mais relevantes foram **NDVI, Índice de Seca, Focos de Queimadas e Precipitação** — todas derivadas diretamente de sensores satelitais.

### Conexão com o tema Space Connect
Esta solução demonstra como **tecnologia espacial (satélites de observação da Terra)** pode ser integrada com **Machine Learning** para salvar vidas em solo. Satélites como o Landsat, CBERS e Sentinel já fornecem esses dados publicamente — nossa solução mostra como transformá-los em **alertas de risco preventivos**.

### Impacto potencial
- Alertas com antecedência de semanas para agências de proteção civil
- Priorização de recursos em regiões de maior risco
- Integração com sistemas de IoT (sensores terrestres + satélite)

---
*Disciplina: Cognitive Computing, Computer Vision and IoT Systems — FIAP 2026*
